# Phase 1: Google Colab UNICEF Data Analysis
### Step 1: Data Upload
Upload the three CSV files (`unicef_indicator_1.csv`, `unicef_indicator_2.csv`, and `unicef_metadata.csv`) into your Google Colab workspace before running this notebook.

In [ ]:
!pip install polars plotnine geopandas scikit-misc -q

import polars as pl
import numpy as np
from plotnine import *
import warnings
warnings.filterwarnings("ignore")

# Load data with Polars
literacy_df = pl.read_csv("unicef_indicator_1.csv", ignore_errors=True)
water_df    = pl.read_csv("unicef_indicator_2.csv", ignore_errors=True)
metadata_df = pl.read_csv("unicef_metadata.csv", ignore_errors=True)

print(f"Loaded: literacy={literacy_df.shape[0]}, water={water_df.shape[0]}, metadata={metadata_df.shape[0]} rows")

### Step 2: Data Cleaning & Pre-processing

In [ ]:
# Clean and Prepare Data
for name in ["literacy_df", "water_df", "metadata_df"]:
    df = eval(name)
    new_cols = {c: c.strip().lower().replace(" ", "_") for c in df.columns}
    exec(f"{name} = df.rename(new_cols)")

obs_col = [c for c in literacy_df.columns if "obs_value" in c or "value" in c][0]
sex_col = [c for c in literacy_df.columns if "sex" in c][0]
country_col = [c for c in literacy_df.columns if "country" in c and "alpha" not in c][0]
alpha_col = [c for c in literacy_df.columns if "alpha_3" in c][0]
year_col = [c for c in literacy_df.columns if "time" in c or "year" in c][0]
w_obs = [c for c in water_df.columns if "obs_value" in c or "value" in c][0]
w_year_col = [c for c in water_df.columns if "time" in c or "year" in c][0]
m_year = [c for c in metadata_df.columns if "year" in c or "time" in c][0]
m_life = [c for c in metadata_df.columns if "life_expectancy" in c.lower() or "life" in c.lower()][0]
m_gdp = [c for c in metadata_df.columns if "gdp" in c.lower()][0]
m_pop = [c for c in metadata_df.columns if "pop" in c.lower()][0]
meta_alpha = [c for c in metadata_df.columns if "alpha_3" in c][0]

literacy_df = literacy_df.with_columns(pl.col(obs_col).cast(pl.Float64, strict=False).alias("obs_value"))
water_df = water_df.with_columns(pl.col(w_obs).cast(pl.Float64, strict=False).alias("obs_value"))

t_latest = literacy_df.filter(pl.col(sex_col).str.to_lowercase() == "total").sort(year_col).group_by(alpha_col).last()


### Step 3: Generate Individual Visualizations
The following blocks generate the plots sequentially and save them to `.png` files for Quarto to collate.

In [ ]:
import geopandas as gpd
import math

world = gpd.read_file("https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip")[["ISO_A3", "geometry", "NAME"]].rename(columns={"ISO_A3": "iso_a3"})
literacy_dict = dict(zip(t_latest[alpha_col].to_list(), t_latest["obs_value"].to_list()))
world["obs_value"] = [literacy_dict.get(code, math.nan) for code in world["iso_a3"]]

p1 = (
    ggplot(world) 
    + geom_map(aes(fill="obs_value"), color="white", size=0.1) 
    + scale_fill_gradient(low="#EF4444", high="#22C55E", na_value="#374151", name="Literacy Rate (%)") 
    + labs(title="Youth Literacy Rates Across the World", subtitle="Latest available data per country")
    + theme_void()
    + theme(plot_title=element_text(size=14, weight="bold"))
)
p1 = p1 + theme(figure_size=(5, 5))
p1.save("vis1_map.png", width=5, height=5, dpi=300)
p1


In [ ]:
bottom10 = t_latest.sort("obs_value").head(10).with_columns(pl.col(country_col).cast(pl.Categorical).alias("country_cat"))
p2 = (
    ggplot(bottom10, aes(x=f"reorder({country_col}, obs_value)", y="obs_value")) 
    + geom_bar(stat="identity", fill="#EF4444", width=0.7) 
    + geom_text(aes(label="obs_value"), format_string="{:.1f}%", ha="left", nudge_y=1, size=8, color="#991B1B")
    + coord_flip() 
    + labs(title="Bottom 10 — Youth Literacy Crisis Nations", x="", y="Literacy Rate (%)")
    + theme_minimal()
    + theme(plot_title=element_text(size=14, weight="bold"))
)
p2 = p2 + theme(figure_size=(5, 5))
p2.save("vis2_bar.png", width=5, height=5, dpi=300)
p2


In [ ]:
meta_clean = (metadata_df.with_columns([pl.col(m_year).cast(pl.Int64, strict=False), pl.col(m_gdp).cast(pl.Float64, strict=False), pl.col(m_pop).cast(pl.Float64, strict=False)]).drop_nulls(subset=[m_year, m_gdp, m_pop]).sort(m_year).group_by(meta_alpha).last().select([pl.col(meta_alpha).alias(alpha_col), pl.col(m_gdp), pl.col(m_pop)]))
scatter_df = t_latest.join(meta_clean, on=alpha_col, how="inner").drop_nulls(subset=["obs_value", m_gdp]).filter(pl.col(m_gdp) > 0)
scatter_df = scatter_df.with_columns(pl.struct([country_col, year_col, "obs_value", m_gdp, m_pop]).map_elements(lambda r: f"{r[country_col]}\n{r['obs_value']:.1f}%" if (r["obs_value"] < 55) or (r[m_gdp] > 70000) or (r[m_pop] > 2e8) else "", return_dtype=pl.Utf8).alias("label"))

p3 = (
    ggplot(scatter_df, aes(x=m_gdp, y="obs_value"))
    + geom_point(aes(size=m_pop), alpha=0.6, color="#3B82F6", fill="#93C5FD", stroke=0.3)
    + geom_smooth(method="lm", color="#EF4444", linetype="dashed", se=True, alpha=0.15)
    + geom_text(aes(label="label"), size=7, color="#0F172A", nudge_y=3.5)
    + scale_x_log10(labels=lambda lst: [f"${v/1000:.0f}k" if v >= 1000 else f"${v:.0f}" for v in lst])
    + labs(title="Literacy vs Economic Development", x="GDP per Capita (USD, log)", y="Literacy (%)")
    + theme_minimal()
    + theme(legend_position="none")
)
p3 = p3 + theme(figure_size=(5, 5))
p3.save("vis3_scatter.png", width=5, height=5, dpi=300)
p3


In [ ]:
water_ts = water_df.with_columns(pl.col(w_year_col).cast(pl.Int64, strict=False).alias("year")).drop_nulls(subset=["year", "obs_value"]).group_by("year").agg(pl.col("obs_value").mean().alias("avg_water")).sort("year")

p4 = (
    ggplot(water_ts, aes(x="year", y="avg_water"))
    + geom_area(fill="#06B6D4", alpha=0.2)
    + geom_line(color="#06B6D4", size=1.2)
    + geom_point(color="#0E7490", size=2.5, alpha=0.8)
    + geom_smooth(method="loess", color="#0891B2", linetype="dashed", se=False, span=0.4)
    + labs(title="Global Water Access Trend", x="Year", y="Basic Water Access (%)")
    + theme_minimal()
)
p4 = p4 + theme(figure_size=(5, 5))
p4.save("vis4_timeseries.png", width=5, height=5, dpi=300)
p4


### Step 4: Generate Quarto Reporting Strings
Run the core cell below to process the `Polars` data structures locally and construct the exact Markdown codes you need to copy-paste directly into your Quarto file. The logic automatically sorts tabs based on the most statistically dense years.

In [ ]:
def get_year_stat(df, time_col, val_col, join_alpha, year, is_sum=False):
    d = df.filter(pl.col(time_col).cast(pl.Int64, strict=False) == year).with_columns(pl.col(val_col).cast(pl.Float64, strict=False)).drop_nulls(subset=[val_col]).unique(subset=[join_alpha], keep='last')
    if d.height == 0: return None
    if is_sum: return d.select(pl.col(val_col).sum()).item()
    return d.select(pl.col(val_col).mean()).item()

# Block 1
block1 = "::: {.panel-tabset}\n\n"
year_counts = literacy_df.filter(pl.col(sex_col).str.to_lowercase() == "total").with_columns(pl.col(year_col).cast(pl.Int64, strict=False)).drop_nulls(subset=[year_col, "obs_value"]).group_by(year_col).agg(pl.col(alpha_col).n_unique().alias("num_countries"))
years_to_show = [2019, 2020, 2021, 2022, 2023, 2024]
top_years = []
for y in years_to_show:
    r = year_counts.filter(pl.col(year_col) == y)
    count = r.select("num_countries").item() if r.height > 0 else 0
    top_years.append((y, count))

for y, count in top_years:
    block1 += f"## {y} ({count} Countries)\n\n"
    l_yr = literacy_df.filter(pl.col(sex_col).str.to_lowercase() == "total", pl.col(year_col).cast(pl.Int64, strict=False) == y).unique(subset=[alpha_col], keep='last')
    total_countries = l_yr.height
    avg_lit = l_yr.select(pl.col("obs_value").mean()).item() if total_countries > 0 else None
    
    female = literacy_df.filter(pl.col(sex_col).str.to_lowercase() == "female", pl.col(year_col).cast(pl.Int64, strict=False) == y).unique(subset=[alpha_col], keep='last').select([alpha_col, country_col, pl.col("obs_value").alias("female_rate")])
    male = literacy_df.filter(pl.col(sex_col).str.to_lowercase() == "male", pl.col(year_col).cast(pl.Int64, strict=False) == y).unique(subset=[alpha_col], keep='last').select([alpha_col, pl.col("obs_value").alias("male_rate")])
    gender = female.join(male, on=alpha_col, how="inner").with_columns((pl.col("male_rate") - pl.col("female_rate")).abs().alias("gap"))
    worst = gender.sort("gap", descending=True).head(1).to_dicts()[0] if gender.height > 0 else None

    tot_pop = get_year_stat(metadata_df, m_year, m_pop, meta_alpha, y, is_sum=True)
    avg_life = get_year_stat(metadata_df, m_year, m_life, meta_alpha, y)
    avg_gdp = get_year_stat(metadata_df, m_year, m_gdp, meta_alpha, y)
    avg_water = get_year_stat(water_df, w_year_col, "obs_value", alpha_col, y)

    tot_pop_str = f"{tot_pop / 1e9:.2f}B" if tot_pop is not None else "Data not available"
    avg_lit_str = f"{avg_lit:.1f}%" if avg_lit is not None else "Data not available"
    avg_life_str = f"{avg_life:.1f}yr" if avg_life is not None else "Data not available"
    avg_gdp_str = f"${avg_gdp:,.0f}" if avg_gdp is not None else "Data not available"
    avg_water_str = f"{avg_water:.1f}%" if avg_water is not None else "Data not available"

    html_content = f"""<div style="font-family: ui-sans-serif, system-ui, sans-serif; background-color: #ffffff; border: 1px solid #e2e8f0; padding: 16px; border-radius: 12px; margin: 0 auto 30px auto; color: #334155; width: 100%; box-shadow: 0 4px 6px -1px rgb(0 0 0 / 0.1);">
<div style="display: grid; grid-template-columns: 1fr 1fr; gap: 12px; margin-bottom: 12px;">
<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 12px 16px; border-radius: 8px; display: flex; flex-direction: column; justify-content: center;">
<div style="font-size: 20px; font-weight: 700; color: #0891b2; margin-bottom: 2px; line-height: 1;">{total_countries if total_countries > 0 else 'Data not available'}</div>
<div style="font-size: 11px; color: #64748b; margin-bottom: 2px; font-weight: 600; text-transform: uppercase;">Data: {y}</div>
<div style="font-size: 14px; font-weight: 500; color: #475569;">Countries Analyzed</div>
</div>
<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 12px 16px; border-radius: 8px; display: flex; flex-direction: column; justify-content: center;">
<div style="font-size: 20px; font-weight: 700; color: #0f172a; margin-bottom: 2px; line-height: 1;">{tot_pop_str}</div>
<div style="font-size: 11px; color: #64748b; margin-bottom: 2px; font-weight: 600; text-transform: uppercase;">Data: {y}</div>
<div style="font-size: 14px; font-weight: 500; color: #475569;">Total Population</div>
</div>
<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 12px 16px; border-radius: 8px; display: flex; flex-direction: column; justify-content: center;">
<div style="font-size: 20px; font-weight: 700; color: #0891b2; margin-bottom: 2px; line-height: 1;">{avg_lit_str}</div>
<div style="font-size: 11px; color: #64748b; margin-bottom: 2px; font-weight: 600; text-transform: uppercase;">Data: {y}</div>
<div style="font-size: 14px; font-weight: 500; color: #475569;">Avg Youth Literacy</div>
</div>
<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 12px 16px; border-radius: 8px; display: flex; flex-direction: column; justify-content: center;">
<div style="font-size: 20px; font-weight: 700; color: #059669; margin-bottom: 2px; line-height: 1;">{avg_life_str}</div>
<div style="font-size: 11px; color: #64748b; margin-bottom: 2px; font-weight: 600; text-transform: uppercase;">Data: {y}</div>
<div style="font-size: 14px; font-weight: 500; color: #475569;">Avg Life Expectancy</div>
</div>
<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 12px 16px; border-radius: 8px; display: flex; flex-direction: column; justify-content: center;">
<div style="font-size: 20px; font-weight: 700; color: #d97706; margin-bottom: 2px; line-height: 1;">{avg_gdp_str}</div>
<div style="font-size: 11px; color: #64748b; margin-bottom: 2px; font-weight: 600; text-transform: uppercase;">Data: {y}</div>
<div style="font-size: 14px; font-weight: 500; color: #475569;">Avg GDP/Capita</div>
</div>
<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 12px 16px; border-radius: 8px; display: flex; flex-direction: column; justify-content: center;">
<div style="font-size: 20px; font-weight: 700; color: #ea580c; margin-bottom: 2px; line-height: 1;">{avg_water_str}</div>
<div style="font-size: 11px; color: #64748b; margin-bottom: 2px; font-weight: 600; text-transform: uppercase;">Data: {y}</div>
<div style="font-size: 14px; font-weight: 500; color: #475569;">Avg Water Access</div>
</div>
</div>"""
    if worst:
        html_content += f"""
<div style="background-color: #fff1f2; border: 1px solid #fecdd3; padding: 12px 16px; border-radius: 8px; display: flex; align-items: center; gap: 10px;">
<div style="font-size: 14px; font-weight: 600; color: #e11d48; letter-spacing: 0.02em;">LARGEST GENDER GAP</div>
<div style="font-size: 18px; font-weight: 700; color: #be123c; margin-left: 4px;">{worst['gap']:.1f}pp</div>
<div style="font-size: 14px; color: #4c1d95; font-weight: 500;">in {worst[country_col]}</div>
</div>"""
    else:
        html_content += f"""
<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 12px 16px; border-radius: 8px; display: flex; align-items: center; gap: 10px;">
<div style="font-size: 14px; font-weight: 600; color: #64748b; letter-spacing: 0.02em;">LARGEST GENDER GAP</div>
<div style="font-size: 18px; font-weight: 700; color: #334155; margin-left: 4px;">Data not available for {y}</div>
</div>"""
    
    html_content += "\n</div>\n\n"
    block1 += html_content

block1 += ":::\n"

# Block 2
block2 = "::: {.panel-tabset}\n\n"
top_years_tabs = [(r[year_col], r["num_countries"]) for r in year_counts.sort("num_countries", descending=True).head(3).to_dicts()]

for y, count in top_years_tabs:
    block2 += f"## {y} ({count} Countries)\n\n"
    yr_data = literacy_df.filter(pl.col(sex_col).str.to_lowercase() == "total", pl.col(year_col).cast(pl.Int64, strict=False) == y).unique(subset=[alpha_col], keep='last')
    
    if yr_data.height == 0:
        block2 += "::: {.callout-note}\n## Insufficient Data\nNot Available or Insufficient data for this year.\n::: \n"
        continue

    top5_map = yr_data.sort("obs_value", descending=True).head(5)
    bottom5_map = yr_data.sort("obs_value").head(5)

    md = "### Highest Literacy (Top 5)\n\n"
    md += "| Country | Record Year | Youth Literacy Rate (%) |\n"
    md += "|:---|:---:|---:|\n"
    for r in top5_map.iter_rows(named=True): md += f"| **{r[country_col]}** | {r[year_col]} | {r['obs_value']:.1f}% |\n"
    md += "\n<br>\n\n"
    md += "### Critical Literacy Deficit (Bottom 5)\n\n"
    md += "| Country | Record Year | Youth Literacy Rate (%) |\n"
    md += "|:---|:---:|---:|\n"
    for r in bottom5_map.iter_rows(named=True): md += f"| **{r[country_col]}** | {r[year_col]} | {r['obs_value']:.1f}% |\n"
    block2 += md + "\n"
block2 += ":::\n"

# Block 3
block3 = "::: {.panel-tabset}\n\n"
for y, count in top_years_tabs:
    block3 += f"## {y} ({count} Countries)\n\n"
    yr_data = literacy_df.filter(pl.col(sex_col).str.to_lowercase() == "total", pl.col(year_col).cast(pl.Int64, strict=False) == y).unique(subset=[alpha_col], keep='last')
    if yr_data.height == 0:
        block3 += "::: {.callout-note}\n## Insufficient Data\nNot Available or Insufficient data for this year.\n::: \n"
        continue

    top5 = yr_data.sort("obs_value", descending=True).head(5)
    bottom5 = yr_data.sort("obs_value").head(5)

    md = "::: {.callout-tip}\n## The Good Story\nThese five nations represent the absolute peak of educational access recorded in the UNICEF database, achieving literally perfect recorded youth literacy. They stand as statistical testaments to robust educational systems, highly structured socio-economic public policies, and unwavering government investments into early childhood infrastructures.\n\n"
    md += "| Country | Record Year | Literacy Rate (%) |\n|:---|:---:|---:|\n"
    for r in top5.iter_rows(named=True): md += f"| **{r[country_col]}** | {r[year_col]} | {r['obs_value']:.1f}% |\n"
    md += ":::\n\n"
    md += "::: {.callout-important}\n## The Bad Story\nConversely, these five extreme outliers sit at the painful epicenter of the global educational crisis. Ravaged heavily by deep-set multidimensional poverty, geographic isolation, and severe regional instability, the vast majority of their youth systematically lack access to foundational learning. Their ongoing humanitarian struggle physically darkens the overarching global map.\n\n"
    md += "| Country | Record Year | Literacy Rate (%) |\n|:---|:---:|---:|\n"
    for r in bottom5.iter_rows(named=True): md += f"| **{r[country_col]}** | {r[year_col]} | {r['obs_value']:.1f}% |\n"
    md += ":::\n"
    block3 += md + "\n"
block3 += ":::\n"

# Block 4
block4 = "::: {.panel-tabset}\n\n"
for y, count in top_years_tabs:
    block4 += f"## {y} ({count} Countries)\n\n"
    yr_meta = metadata_df.filter(pl.col(m_year).cast(pl.Int64, strict=False) == y).with_columns([pl.col(m_gdp).cast(pl.Float64, strict=False),pl.col(m_pop).cast(pl.Float64, strict=False)]).drop_nulls(subset=[m_gdp, m_pop]).unique(subset=[meta_alpha], keep='last')
    yr_data = literacy_df.filter(pl.col(sex_col).str.to_lowercase() == "total", pl.col(year_col).cast(pl.Int64, strict=False) == y).unique(subset=[alpha_col], keep='last')
    top15 = yr_data.join(yr_meta, left_on=alpha_col, right_on=meta_alpha, how="inner").drop_nulls(subset=["obs_value", m_gdp]).sort(m_gdp, descending=True).head(15).with_columns(pl.col(m_pop).cast(pl.Int64, strict=False))
    
    if top15.height == 0:
        block4 += "::: {.callout-note}\n## Insufficient GDP Data\nNot Available or Insufficient data for this year.\n::: \n"
        continue

    md = "| Country | Record Year | GDP per Capita (USD) | Total Population | Youth Literacy (%) |\n"
    md += "|:---|:---:|---:|---:|---:|\n"
    for r in top15.iter_rows(named=True): md += f"| **{r[country_col]}** | {r[year_col]} | ${r[m_gdp]:,.0f} | {r[m_pop]:,} | {r['obs_value']:.1f}% |\n"
    block4 += md + "\n"
block4 += ":::\n"

# Block 5
block5 = "::: {.panel-tabset}\n\n"
for y, count in top_years_tabs:
    block5 += f"## {y} ({count} Countries)\n\n"
    yr_water = water_df.filter(pl.col(w_year_col).cast(pl.Int64, strict=False) == y).unique(subset=[alpha_col], keep='last')
    if yr_water.height == 0:
        block5 += "::: {.callout-note}\n## Insufficient Water Data\nNot Available or Insufficient data for this year.\n::: \n"
        continue
    
    top5_w = yr_water.sort("obs_value", descending=True).head(5)
    bottom5_w = yr_water.sort("obs_value").head(5)
    
    md = "::: {.callout-tip}\n## Universal Water Access (Top 5)\nThese nations have achieved 100% basic water access for their populations, ensuring a critical foundation for health and education.\n\n"
    md += "| Country | Record Year | Basic Water Access (%) |\n|:---|:---:|---:|\n"
    for r in top5_w.iter_rows(named=True): md += f"| **{r[country_col]}** | {r[w_year_col]} | {r['obs_value']:.1f}% |\n"
    md += ":::\n\n"
    md += "::: {.callout-important}\n## Critical Water Insecurity (Bottom 5)\nIn contrast, these regions face severe water scarcity, with access rates as low as 8.5%, forcing youth into resource-gathering roles rather than schooling.\n\n"
    md += "| Country | Record Year | Basic Water Access (%) |\n|:---|:---:|---:|\n"
    for r in bottom5_w.iter_rows(named=True): md += f"| **{r[country_col]}** | {r[w_year_col]} | {r['obs_value']:.1f}% |\n"
    md += ":::\n"
    block5 += md + "\n"
block5 += ":::\n"


### Step 5: Quarto Assembly Render
Install the Quarto CLI inside the Colab environment natively compiling your final `.qmd` file. Ensure `UNICEF_Polars_Report.qmd` is uploaded along with your CSV files before executing this block.

In [ ]:
!wget https://github.com/quarto-dev/quarto-cli/releases/download/v1.4.553/quarto-1.4.553-linux-amd64.deb -q
!dpkg -i quarto-1.4.553-linux-amd64.deb > /dev/null 2>&1
!quarto --version
!quarto render UNICEF_Polars_Report.qmd